# Unified Synthetic Data Generation

Generate synthetic EIT, EMG, and ventilator data from one YAML configuration. The default YAML generates EIT only with timestamped output folders. Enable EMG and ventilator generation in YAML, or use the optional cell at the end, when ReSurfEMG is installed.

## Imports

In [ ]:
import os
from dataclasses import replace

import numpy as np

import m3resp.synthetic
from m3resp.synthetic import (
    generate_synthetic_dataset,
    load_synthetic_generator_config,
)

generator_dir = os.path.dirname(os.path.abspath(m3resp.synthetic.__file__))

## Configure A Dataset With YAML

The notebook loads `synthetic_generator_config.yaml` from the installed `m3resp.synthetic` package module. Edit that YAML file to change durations, sampling rates, drift, EIT waveform details, lung-template shape, Medibus channel settings, event markers, timestamped output behavior, and modality toggles.


In [2]:
config_path = os.path.join(generator_dir, "synthetic_generator_config.yaml")
config = load_synthetic_generator_config(config_path)

dataset = generate_synthetic_dataset(config)
run_output_dir = dataset.provenance["output_dir"]
run_output_dir


'/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100254'

## Inspect Outputs

In [3]:
dataset.eit.metadata

{'modality': 'eit',
 'vendor': 'draeger',
 'format_name': 'original',
 'frame_size_bytes': 4358,
 'time_units': 'seconds',
 'signal_units': 'relative impedance (a.u.)',
 'component_labels': ['baseline', 'breathing', 'cardiac', 'drift', 'noise']}

In [4]:
dataset.eit.array.shape, dataset.eit.sample_frequency, dataset.eit.paths


((1200, 32, 32),
 20.0,
 {'npy': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100254/m3resp_demo_eit_pixels.npy',
  'csv': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100254/m3resp_demo_eit_global.csv',
  'components_npz': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100254/m3resp_demo_eit_components.npz',
  'native': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100254/m3resp_demo_eit_draeger.bin'})

In [5]:
sorted(os.listdir(run_output_dir))


['m3resp_demo_eit_components.npz',
 'm3resp_demo_eit_draeger.bin',
 'm3resp_demo_eit_global.csv',
 'm3resp_demo_eit_pixels.npy',
 'm3resp_demo_emg.Poly5',
 'm3resp_demo_emg.csv',
 'm3resp_demo_emg.npy',
 'm3resp_demo_metadata.json',
 'm3resp_demo_ventilator.Poly5',
 'm3resp_demo_ventilator.csv',
 'm3resp_demo_ventilator.npy',
 'm3resp_demo_ventilator_p_mus.npy']

## Inspect The Drift Component

In [6]:
components = np.load(dataset.eit.paths["components_npz"])
components.files

['baseline', 'breathing', 'drift', 'cardiac', 'noise']

In [7]:
components["drift"][:10]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## Optional EMG And Ventilator Generation

This cell reuses the YAML-loaded values and only changes the modality toggles in memory. It runs when ReSurfEMG is installed. Portable `.npy` and `.csv` exports are written by default; set `write_native_outputs: true` in YAML only when your installed ReSurfEMG version exposes a native synthetic recording writer.

In [8]:
multimodal_config = replace(
    config,
    basename=f"{config.basename}_multimodal",
    generate_eit=True,
    generate_emg=True,
    generate_ventilator=True,
    write_native_outputs=False,
)

try:
    multimodal_dataset = generate_synthetic_dataset(multimodal_config)
    display(
        {
            "run_output_dir": multimodal_dataset.provenance["output_dir"],
            "emg": multimodal_dataset.emg.paths,
            "ventilator": multimodal_dataset.ventilator.paths,
        }
    )
except RuntimeError as exc:
    print(exc)


{'run_output_dir': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100257',
 'emg': {'npy': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100257/m3resp_demo_multimodal_emg.npy',
  'csv': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100257/m3resp_demo_multimodal_emg.csv'},
 'ventilator': {'npy': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100257/m3resp_demo_multimodal_ventilator.npy',
  'csv': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100257/m3resp_demo_multimodal_ventilator.csv',
  'p_mus_npy': '/home/mahyart/Desktop/github_repos/m3resp-org/m3resp/data/source/synthetic_demo/20260610_100257/m3resp_demo_multimodal_ventilator_p_mus.npy'}}